# 11 CNN Final Selected Model

Only this notebook evaluates the frozen held-out test set, after notebook 10 has selected hyperparameters.


## 1. Package Setup


In [ ]:
# Purpose: Package installs are documented but not run automatically during this static refactor.
# %pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib soundfile librosa


## 2. Load Selected Hyperparameters and Test Cache


In [ ]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the project files, cached MFCC features, manifests, models, figures,
# and metric outputs live in Google Drive during Colab runs. The /content/drive path
# only represents the real MyDrive files after drive.mount("/content/drive") succeeds.
# Important: the project root should be the folder that contains Data, Model Variants,
# and outputs. For this project, that expected Colab folder is the INM701 folder below.
COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount(str(COLAB_DRIVE_MOUNT_POINT))
    print("Google Colab detected. Google Drive mounted.")

    current_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT")
    current_data_dir_exists = bool(current_project_root) and (Path(current_project_root) / "Data").exists()
    expected_data_dir_exists = (EXPECTED_COLAB_PROJECT_ROOT / "Data").exists()

    # Purpose: Keep a valid user-provided project root, but repair stale runtime state
    # if a previous cell pointed INTRO_AI_PROJECT_ROOT somewhere that does not contain Data.
    if current_data_dir_exists:
        print("INTRO_AI_PROJECT_ROOT already points to a folder with Data, so it was kept.")
    elif expected_data_dir_exists:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.")
    elif not current_project_root:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT was not set, so it now points to the expected INM701 folder.")
    else:
        print("INTRO_AI_PROJECT_ROOT was kept, but Data was not found there or in the expected INM701 folder.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

active_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT", "not set")
print("INTRO_AI_PROJECT_ROOT:", active_project_root)
if active_project_root != "not set":
    active_project_root = Path(active_project_root)
    print("Project root exists:", active_project_root.exists())
    print("Expected Data folder:", active_project_root / "Data")
    print("Data folder exists:", (active_project_root / "Data").exists())


In [ ]:
# Purpose: Later notebooks load the one CNN-ready cache made by notebook 00. They do not
# rescan folders, regenerate splits, extract MFCCs, fit scalers, or recalculate class weights.
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Conv2D, Dense, Dropout, GlobalAveragePooling2D, Input, MaxPooling2D
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
CONFIRMATION_SEEDS = [42, 123, 2026]


def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cnn"
CACHE_DIR = OUTPUT_DIR / "cache"
MANIFESTS_DIR = OUTPUT_DIR / "manifests"
CONFIGS_DIR = OUTPUT_DIR / "configs"
TABLES_DIR = OUTPUT_DIR / "tables"
HISTORIES_DIR = OUTPUT_DIR / "histories"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
PILOT_LEGACY_DIR = OUTPUT_DIR / "pilot_legacy"
# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [OUTPUT_DIR, CACHE_DIR, MANIFESTS_DIR, CONFIGS_DIR, TABLES_DIR, HISTORIES_DIR, METRICS_DIR, FIGURES_DIR, MODELS_DIR, PILOT_LEGACY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    # Purpose: Stops the notebook early with a clear message when a required upstream artifact is missing.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


def load_json(path):
    # Purpose: Keeps the load_json helper isolated so later notebook cells can call it consistently.
    with open(require_file(path), "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(payload, path):
    # Purpose: Keeps the save_json helper isolated so later notebook cells can call it consistently.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def load_class_weights():
    # Purpose: Keeps the load_class_weights helper isolated so later notebook cells can call it consistently.
    payload = load_json(CONFIGS_DIR / "class_weights.json")
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    CONFIGS_DIR / "class_weights.json",
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy", mmap_mode="r")[..., np.newaxis]
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy", mmap_mode="r")[..., np.newaxis]
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
feature_config = load_json(CACHE_DIR / "feature_config.json")
CLASS_WEIGHTS = load_class_weights()

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training cache arrays, labels and metadata are not row-aligned.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation cache arrays, labels and metadata are not row-aligned.")

print("Loaded CNN cache:", CACHE_DIR)
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)

selected_hyperparameters_path = CONFIGS_DIR / "selected_hyperparameters.json"
if not selected_hyperparameters_path.exists():
    raise FileNotFoundError(
        f"Required selected hyperparameters file not found: {selected_hyperparameters_path}. "
        "Run 10_CNN_HPO_Comparison.ipynb first so selected_hyperparameters.json can be created "
        "from confirmation results or validation-only HPO evidence."
    )
selected_payload = load_json(selected_hyperparameters_path)
final_config = selected_payload["configuration"]
X_test = np.load(CACHE_DIR / "X_test.npy", mmap_mode="r")[..., np.newaxis]
y_test = np.load(CACHE_DIR / "y_test.npy")
test_metadata = pd.read_csv(CACHE_DIR / "test_metadata.csv")
print("Loaded selected_hyperparameters.json and held-out test cache.")


## 3. Keras Utilities


In [ ]:
# Purpose: Shared CNN utilities for validation-only model selection.
def normalise_config(config):
    # Purpose: Converts JSON/CSV values into the types expected by the CNN builder.
    normalised = dict(config)
    normalised["conv_filters"] = [int(value) for value in normalised["conv_filters"]]
    normalised["kernel_size"] = [int(value) for value in normalised["kernel_size"]]
    normalised["dense_units"] = int(normalised["dense_units"])
    normalised["activation"] = str(normalised.get("activation", "relu"))
    normalised["optimizer"] = str(normalised.get("optimizer", "adam"))
    normalised["dropout"] = float(normalised["dropout"])
    normalised["learning_rate"] = float(normalised["learning_rate"])
    normalised["batch_size"] = int(normalised["batch_size"])
    normalised["loss"] = normalised.get("loss", "binary_crossentropy")
    normalised["threshold"] = float(normalised.get("threshold", THRESHOLD))
    return normalised


def config_json(config):
    # Purpose: Creates a stable representation for deduplication and deterministic seeds.
    return json.dumps(normalise_config(config), sort_keys=True)


def config_key(config):
    return hashlib.sha256(config_json(config).encode("utf-8")).hexdigest()


def seed_from_config(config, base_seed=RANDOM_STATE):
    digest = hashlib.sha256(f"{base_seed}:{config_json(config)}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)


def set_global_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))
    os.environ["PYTHONHASHSEED"] = str(int(seed))


def make_optimizer(config):
    name = str(config["optimizer"]).lower()
    learning_rate = float(config["learning_rate"])
    if name == "adam":
        return tf.keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "rmsprop":
        return tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    if name in ["sgd", "sgd_momentum"]:
        return tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    raise ValueError(f"Unsupported optimizer: {config['optimizer']}")


def build_cnn_model(config, input_shape):
    # Purpose: Learns local patterns from the coefficient-by-time MFCC map.
    config = normalise_config(config)
    model = Sequential(name="mfcc_cnn")
    model.add(Input(shape=tuple(input_shape)))
    for filters in config["conv_filters"]:
        model.add(
            Conv2D(
                filters=filters,
                kernel_size=tuple(config["kernel_size"]),
                padding="same",
                strides=(1, 1),
                activation=config["activation"],
            )
        )
        model.add(MaxPooling2D(pool_size=(2, 2)))
        if config["dropout"] > 0:
            model.add(Dropout(config["dropout"]))
    # Global pooling controls parameter growth when the number of MFCC time frames changes.
    model.add(GlobalAveragePooling2D())
    model.add(Dense(config["dense_units"], activation=config["activation"]))
    if config["dropout"] > 0:
        model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(loss=config["loss"], optimizer=make_optimizer(config), metrics=["accuracy"])
    return model


def train_and_evaluate_config(config, run_seed, run_name, verbose=0):
    # Purpose: Uses training data for fitting and validation data for selection; never test data.
    config = normalise_config(config)
    tf.keras.backend.clear_session()
    set_global_seed(run_seed)
    model = build_cnn_model(config, X_train.shape[1:])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        )
    ]
    start = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=verbose,
    )
    runtime = time.perf_counter() - start
    probability = model.predict(X_validation, batch_size=config["batch_size"], verbose=0).ravel()
    pred = (probability >= config["threshold"]).astype(int)
    row = {
        "configuration": config,
        "config_json": config_json(config),
        "config_key": config_key(config),
        "seed": int(run_seed),
        "validation_macro_f1": float(f1_score(y_validation, pred, average="macro", zero_division=0)),
        "validation_binary_f1_synthetic": float(f1_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_accuracy": float(accuracy_score(y_validation, pred)),
        "validation_precision_synthetic": float(precision_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_recall_synthetic": float(recall_score(y_validation, pred, pos_label=1, zero_division=0)),
        "best_validation_loss": float(np.min(history.history["val_loss"])),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_trained": int(len(history.history["loss"])),
        "runtime_seconds": float(runtime),
        "run_name": run_name,
    }
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    return row, history_df


def flat_result(row, extra=None):
    # Purpose: Makes each CNN configuration readable in CSV result tables.
    extra = extra or {}
    config = normalise_config(row["configuration"])
    flat = dict(extra)
    flat.update({
        "conv_filters": json.dumps(config["conv_filters"]),
        "kernel_size": json.dumps(config["kernel_size"]),
        "dense_units": config["dense_units"],
        "activation": config["activation"],
        "optimizer": config["optimizer"],
        "dropout": config["dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "loss": config["loss"],
        "threshold": config["threshold"],
        "config_json": row["config_json"],
        "config_key": row["config_key"],
        "seed": row["seed"],
        "validation_macro_f1": row["validation_macro_f1"],
        "validation_binary_f1_synthetic": row["validation_binary_f1_synthetic"],
        "validation_accuracy": row["validation_accuracy"],
        "validation_precision_synthetic": row["validation_precision_synthetic"],
        "validation_recall_synthetic": row["validation_recall_synthetic"],
        "best_validation_loss": row["best_validation_loss"],
        "best_epoch": row["best_epoch"],
        "epochs_trained": row["epochs_trained"],
        "runtime_seconds": row["runtime_seconds"],
    })
    return flat


def select_best(rows):
    return max(rows, key=lambda row: (row["validation_macro_f1"], -row["best_validation_loss"]))


## 4. Final Held-Out Test Evaluation


In [ ]:
# Purpose: 4. Final Held-Out Test Evaluation.
RUN_FINAL_TEST_EVALUATION = True
test_metrics_path = METRICS_DIR / "11_cnn_test_metrics.json"
confusion_matrix_path = METRICS_DIR / "11_cnn_confusion_matrix.csv"
language_subgroup_path = METRICS_DIR / "11_cnn_language_subgroup_metrics.csv"
tts_subgroup_path = METRICS_DIR / "11_cnn_tts_generator_subgroup_metrics.csv"
def subgroup_metrics(metadata, y_true, y_pred, group_column, min_rows=5):
    # Purpose: Keeps the subgroup_metrics helper isolated so later notebook cells can call it consistently.
    if group_column not in metadata.columns:
        return pd.DataFrame()
    working = metadata.copy().reset_index(drop=True)
    working["y_true"] = y_true
    working["y_pred"] = y_pred
    rows = []
    # Purpose: Runs the controlled trial once for each candidate value of the tested hyperparameter.
    for value, group in working.groupby(group_column):
        if len(group) >= min_rows:
            rows.append({group_column: value, "files": int(len(group)), "macro_f1": f1_score(group["y_true"], group["y_pred"], average="macro", zero_division=0), "binary_f1_synthetic": f1_score(group["y_true"], group["y_pred"], pos_label=1, zero_division=0), "accuracy": accuracy_score(group["y_true"], group["y_pred"])})
    return pd.DataFrame(rows)
if RUN_FINAL_TEST_EVALUATION:
    tf.keras.backend.clear_session()
    set_global_seed(RANDOM_STATE)
    final_model = build_cnn_model(final_config, X_train.shape[1:])
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA, restore_best_weights=True)]
    final_model.fit(X_train, y_train, validation_data=(X_validation, y_validation), epochs=MAX_EPOCHS, batch_size=final_config["batch_size"], class_weight=CLASS_WEIGHTS, callbacks=callbacks, verbose=2)
    test_probability = final_model.predict(X_test, batch_size=final_config["batch_size"], verbose=0).ravel()
    test_pred = (test_probability >= final_config["threshold"]).astype(int)
    metrics = {"test_macro_f1": f1_score(y_test, test_pred, average="macro", zero_division=0), "test_binary_f1_synthetic": f1_score(y_test, test_pred, pos_label=1, zero_division=0), "test_binary_f1_bona_fide": f1_score(y_test, test_pred, pos_label=0, zero_division=0), "test_accuracy": accuracy_score(y_test, test_pred), "test_precision_synthetic": precision_score(y_test, test_pred, pos_label=1, zero_division=0), "test_recall_synthetic": recall_score(y_test, test_pred, pos_label=1, zero_division=0)}
    save_json(metrics, test_metrics_path)
    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
    # Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same
    # Purpose: outputs.
    pd.DataFrame(cm, index=[CLASS_NAMES[0], CLASS_NAMES[1]], columns=[CLASS_NAMES[0], CLASS_NAMES[1]]).to_csv(confusion_matrix_path)
    language_df = subgroup_metrics(test_metadata, y_test, test_pred, "language")
    tts_df = subgroup_metrics(test_metadata, y_test, test_pred, "tts_generator")
    if not language_df.empty:
        language_df.to_csv(language_subgroup_path, index=False)
    if not tts_df.empty:
        tts_df.to_csv(tts_subgroup_path, index=False)
    display(pd.DataFrame([metrics]))
else:
    print("RUN_FINAL_TEST_EVALUATION is False. [RESULT TO BE INSERTED AFTER FINAL RUN]")


## 5. Checks


In [ ]:
# Purpose: Runs this notebook step and prints or saves the resulting intermediate output for review.
display(pd.DataFrame([("loads_selected_hyperparameters_json", True), ("only_notebook_allowed_to_compute_test_metrics", True), ("do_not_retune_after_test", True)], columns=["check", "value"]))
